In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

DB = Path("../data/processed/oral_oncology.db")
conn = sqlite3.connect(DB)

def run(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

In [2]:
# Q1. Population KPIs
run("""
SELECT
    COUNT(*) AS total_patients,
    SUM(oral_cancer_positive) AS cancer_positive_count,
    ROUND(100.0 * AVG(oral_cancer_positive), 2) AS cancer_prevalence_pct,
    SUM(CASE WHEN smoking_status = 'Current' THEN 1 ELSE 0 END) AS current_smokers,
    SUM(CASE WHEN alcohol_use = 'Heavy' THEN 1 ELSE 0 END) AS heavy_drinkers,
    SUM(CASE WHEN hpv_status = 'Positive' THEN 1 ELSE 0 END) AS hpv_positive
FROM patients;
""")

,total_patients,cancer_positive_count,cancer_prevalence_pct,current_smokers,heavy_drinkers,hpv_positive
0,5000,376,7.52,1083,536,499


In [3]:
# Q4. Referral delay by insurance type
run("""
SELECT
    p.insurance_type,
    COUNT(r.referral_id) AS n_completed_referrals,
    ROUND(AVG(r.days_to_specialist_visit), 1) AS avg_days_to_specialist
FROM patients p
JOIN referrals r ON p.patient_id = r.patient_id
WHERE r.referral_status = 'Completed'
GROUP BY p.insurance_type
ORDER BY avg_days_to_specialist DESC;
""")

,insurance_type,n_completed_referrals,avg_days_to_specialist
0,Uninsured,25,25.7
1,Medicaid,85,25.1
2,Private,112,23.3
3,Medicare,334,23.0


In [4]:
# Q8. Top 10 priority outreach patients
run("""
WITH risk_score AS (
    SELECT
        p.patient_id, p.age, p.smoking_status, p.alcohol_use, p.hpv_status,
        (CASE WHEN p.smoking_status = 'Current' THEN 3 WHEN p.smoking_status = 'Former' THEN 1 ELSE 0 END
       + CASE WHEN p.alcohol_use = 'Heavy' THEN 2 WHEN p.alcohol_use = 'Moderate' THEN 1 ELSE 0 END
       + CASE WHEN p.hpv_status = 'Positive' THEN 2 ELSE 0 END
       + CASE WHEN p.age >= 60 THEN 1 ELSE 0 END) AS risk_score
    FROM patients p
)
SELECT * FROM risk_score
ORDER BY risk_score DESC
LIMIT 10;
""")

,patient_id,age,smoking_status,alcohol_use,hpv_status,risk_score
0,P00126,68,Current,Heavy,Positive,8
1,P00618,87,Current,Heavy,Positive,8
2,P00785,83,Current,Heavy,Positive,8
3,P01361,81,Current,Heavy,Positive,8
4,P02002,84,Current,Heavy,Positive,8
5,P02219,66,Current,Heavy,Positive,8
6,P02891,60,Current,Heavy,Positive,8
7,P03370,85,Current,Heavy,Positive,8
8,P03389,68,Current,Heavy,Positive,8
9,P03410,72,Current,Heavy,Positive,8
